# EC2 + S3 + Bedrock + Competition Submission
Team-facing operational notebook. Run from the repository root on the EC2 instance.


In [ ]:
from pathlib import Path
import os, json, sys
import pandas as pd
import boto3
sys.path.insert(0, str(Path('src').resolve()))
from cyber_exclusion.io import read_csv
from cyber_exclusion.profile import profile_training_data
from cyber_exclusion.pipeline import train, predict


## 1. Paths and AWS configuration
Adjust only these variables for your EC2 workspace.


In [ ]:
TRAIN = 'data/training-data.csv'
VALIDATION = 'data/validation-data.csv'
CONFIG = 'config/default.yaml'
TRAIN_OUT = 'artifacts/run_001'
PREDICT_OUT = 'artifacts/submission_001'
REGION = 'us-east-1'
BUCKET = 'my-competition-bucket'
RAW_PREFIX = '2026/data/raw_scan_data/'


## 2. Verify the EC2 IAM identity


In [ ]:
sts = boto3.client('sts', region_name=REGION)
identity = sts.get_caller_identity()
print(identity['Arn'])


## 3. Verify S3 access


In [ ]:
s3 = boto3.client('s3', region_name=REGION)
resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=RAW_PREFIX, MaxKeys=5)
for obj in resp.get('Contents', []):
    print(obj['Key'])


## 4. Profile competition manifests


In [ ]:
tr = read_csv(TRAIN)
va = read_csv(VALIDATION)
print(profile_training_data(tr))
print('validation rows:', len(va))
print('validation duplicate keys:', int(va.duplicated(['scan_id','asset']).sum()))
display(tr.head())
display(va.head())


## 5. Train
This uses the S3 raw root in `config/default.yaml`. The raw JSON required for each row is downloaded through Boto3 into the run cache.


In [ ]:
metrics = train(TRAIN, CONFIG, TRAIN_OUT)
print(json.dumps(metrics, indent=2))


## 6. Score validation and generate submission


In [ ]:
scored = predict(VALIDATION, f'{TRAIN_OUT}/model.joblib', CONFIG, PREDICT_OUT)
display(scored.sort_values('y_pred', ascending=False))
submission = pd.read_csv(f'{PREDICT_OUT}/submission.csv')
display(submission)


## 7. Submission assertions


In [ ]:
assert list(submission.columns) == ['scan_id','asset','y_pred']
assert not submission.duplicated(['scan_id','asset']).any()
assert len(submission) == len(va)
assert submission[['scan_id','asset']].astype(str).reset_index(drop=True).equals(va[['scan_id','asset']].astype(str).reset_index(drop=True))
assert pd.to_numeric(submission['y_pred'], errors='coerce').notna().all()
print('Submission contract: PASS')


## 8. Optional Claude / Bedrock evidence explanation
Set `BEDROCK_MODEL_ID` in the shell or notebook environment. Claude is used as an audit narrator only; it does not change `y_pred`.


In [ ]:
model_id = os.environ.get('BEDROCK_MODEL_ID')
if not model_id:
    print('BEDROCK_MODEL_ID is not set; skipping Bedrock demo.')
else:
    from cyber_exclusion.bedrock_explainer import explain_with_bedrock
    row = scored.sort_values('y_pred', ascending=False).iloc[0].to_dict()
    keep = ['client_id','scan_id','asset','y_pred','p_exclude','conformal_set','decision','evidence_coverage','trust_score','review_reason']
    evidence = {k: row.get(k) for k in keep if k in row}
    print(explain_with_bedrock(model_id, REGION, evidence))


## 9. Final file
Upload `artifacts/submission_001/submission.csv` to the competition portal after the assertions above pass.
